# Notebook to test GPU training on cloud computing

This notebook expects only a single CUDA or MPS GPU to be available.

In [1]:
from time import time

import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"torch.accelerater.current_accelerator() gives {device} device")
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#print(f"CUDA device is {device}")

torch.accelerater.current_accelerator() gives mps device


# First benchmark: matrix multiplication

In [3]:
B, N = 64, 2048
num_repeat = 5

In [4]:
# First: CPU
M1 = torch.randn(B, N, N).to(device='cpu')
M2 = torch.randn(B, N, N).to(device='cpu')

# JIT compilation happens
res = torch.bmm(M1, M2)

t1 = time()
for _ in range(num_repeat):
    res = torch.bmm(M1, M2)
t2 = time()
print(f"Time per matrix multiplication on cpu is {(t2-t1)*1000/num_repeat:.2f} ms")

Time per matrix multiplication on cpu is 2352.24 ms


In [5]:
# Second: MPS or CUDA device
if device != 'cpu':
    M1 = torch.randn(B, N, N).to(device)
    M2 = torch.randn(B, N, N).to(device)

    # JIT compilation happens
    res = torch.bmm(M1, M2)

    t1 = time()
    for _ in range(num_repeat):
        res = torch.bmm(M1, M2)
    if device == 'mps':
        torch.mps.synchronize()
    elif device == 'cuda':
        torch.cuda.synchronize()
    else:
        raise ValueError(f"Device {device} not expected")
    t2 = time()
    print(f"Time per matrix multiplication on {device} is {(t2-t1)*1000/num_repeat:.2f} ms")

Time per matrix multiplication on mps is 827.36 ms


# Second benchmark: train on MNIST 

In [6]:
batch_size = 64

In [7]:
import gzip
from pathlib import Path
import pickle
import requests

from torch.utils.data import TensorDataset, DataLoader

def get_MNIST_dataloaders():
    DATA_PATH = Path("data")
    PATH = DATA_PATH / "mnist"
    PATH.mkdir(parents=True, exist_ok=True)
    URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
    FILENAME = "mnist.pkl.gz"
    if not (PATH / FILENAME).exists():
        content = requests.get(URL + FILENAME).content
        (PATH / FILENAME).open("wb").write(content)
    with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_val, y_val), _) = pickle.load(f, encoding="latin-1")
    x_train, y_train = map(torch.tensor, (x_train, y_train))
    x_val, y_val = map(torch.tensor, (x_val, y_val))

    train_ds = TensorDataset(x_train, y_train)
    val_ds = TensorDataset(x_val, y_val)
    train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=64)
    return train_dl, val_dl

In [8]:
train_dl, val_dl = get_MNIST_dataloaders()

In [9]:
class MNISTMLP(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.linear_layers = nn.ModuleList()
        self.linear_layers.append(nn.Linear(784, num_hiddens))
        self.linear_layers.append(nn.Linear(num_hiddens, num_hiddens))
        self.linear_layers.append(nn.Linear(num_hiddens, num_hiddens))
        self.linear_layers.append(nn.Linear(num_hiddens, num_hiddens))
        self.linear_out = nn.Linear(num_hiddens, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        for layer in self.linear_layers:
            x = self.relu(layer(x))
        return self.linear_out(x)

In [10]:
num_epochs = 5
lr = 0.05

In [11]:
def step(model, opt, X, y):
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    return loss.item()

def val_stats(model, X, y):
    with torch.no_grad():
        logits = model(X)
        loss = F.cross_entropy(logits, y)
    return loss.item()

In [12]:
# First: timing for CPU

model = MNISTMLP(128).to('cpu')
opt = torch.optim.SGD(model.parameters(), lr)

t1 = time()

for epoch in range(num_epochs):
    for X, y in train_dl:
        X = X.to('cpu')
        y = y.to('cpu')
        _ = step(model, opt, X, y)

    for X, y in val_dl:
        X = X.to('cpu')
        y = y.to('cpu')
        _ = val_stats(model, X, y)

t2 = time()

print(f"Time to train MNIST on cpu is {(t2-t1):.2f} seconds")

Time to train MNIST on cpu is 3.97 seconds


In [13]:
# Second: timing for mps and cuda
if device != 'cpu':
    model = MNISTMLP(128).to(device)
    opt = torch.optim.SGD(model.parameters(), lr)

    t1 = time()

    for epoch in range(num_epochs):
        for X, y in train_dl:
            X = X.to(device)
            y = y.to(device)
            _ = step(model, opt, X, y)

        for X, y in val_dl:
            X = X.to(device)
            y = y.to(device)
            _ = val_stats(model, X, y)

    if device == 'mps':
        torch.mps.synchronize()
    elif device == 'cuda':
        torch.cuda.synchronize()
    else:
        raise ValueError(f"Device {device} not expected")

    t2 = time()

    print(f"Time to train MNIST on {device} is {(t2-t1):.2f} seconds")

Time to train MNIST on mps is 10.90 seconds
